In [1]:
from pathlib import Path
import copy
import json
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights,
    convnext_tiny,
    ConvNeXt_Tiny_Weights,
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights,
)

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)

C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Confirming CUDA is being used

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", GPU_NAME)
print("Device:", DEVICE)

PyTorch: 2.14.0+cu130
CUDA: 13.0
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
Device: cuda


## 2. Reproducibility and project paths

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models" / "cnn_benchmark"
REPORT_DIR = PROJECT_ROOT / "reports" / "cnn_benchmark"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development


## 3. Loading the baseline split

In [4]:
df = pd.read_csv(
    DATA_DIR / "interim" / "compcars_resnet50_baseline_split.csv"
)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

NUM_CLASSES = df["target"].nunique()

class_names = (
    df[["target", "class_name"]]
    .drop_duplicates()
    .sort_values("target")["class_name"]
    .tolist()
)

print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Classes:", NUM_CLASSES)

Train / Val / Test: 1600 200 200
Classes: 20


## 4. Configuring CompCars image path

In [5]:
DEFAULT_COMPCARS = (
    r"G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J"
    r"\capstone-vehicle-metadata\data\compcars_kaggle\extracted"
)

ROOT = Path(os.environ.get("COMPCARS_EXTRACTED", DEFAULT_COMPCARS))

if not ROOT.exists():
    raise FileNotFoundError(f"CompCars path does not exist:\n{ROOT}")

def resolve_local_path(original_path):
    text = str(original_path).replace("\\", "/")

    if "/extracted/" in text:
        relative = text.split("/extracted/", 1)[1]
    elif "/image/" in text:
        relative = "image/" + text.split("/image/", 1)[1]
    else:
        return Path(original_path)

    return ROOT / Path(*relative.split("/"))

for part in [train_df, val_df, test_df]:
    part["local_file_path"] = part["file_path"].map(resolve_local_path)

sample = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True,
).sample(30, random_state=SEED)

if not sample["local_file_path"].map(Path.exists).all():
    raise FileNotFoundError("Some image paths do not resolve.")

print("Image path mapping looks good.")

Image path mapping looks good.


## 5. Preprocessing and dataset

In [6]:
IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.80, 1.0),
        ratio=(0.85, 1.15),
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
    ),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CompCarsDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(row["local_file_path"]).convert("RGB")
        return self.transform(image), int(row["target"])

## 6. DataLoaders

In [7]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(
    CompCarsDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    CompCarsDataset(val_df, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    CompCarsDataset(test_df, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Batch size:", BATCH_SIZE)

Batch size: 16


## 7. Building the CNN architectures

In [8]:
def build_model(model_name):
    if model_name == "efficientnet_b0":
        model = efficientnet_b0(
            weights=EfficientNet_B0_Weights.DEFAULT
        )

        for p in model.parameters():
            p.requires_grad = False

        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(
            in_features,
            NUM_CLASSES,
        )

        head_parameters = model.classifier.parameters()

        def unfreeze_stage2():
            for p in model.features[-1].parameters():
                p.requires_grad = True

        stage2_lr = 1e-4

    elif model_name == "convnext_tiny":
        model = convnext_tiny(
            weights=ConvNeXt_Tiny_Weights.DEFAULT
        )

        for p in model.parameters():
            p.requires_grad = False

        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(
            in_features,
            NUM_CLASSES,
        )

        head_parameters = model.classifier.parameters()

        def unfreeze_stage2():
            for p in model.features[-1].parameters():
                p.requires_grad = True

        stage2_lr = 5e-5

    elif model_name == "mobilenet_v3_large":
        model = mobilenet_v3_large(
            weights=MobileNet_V3_Large_Weights.DEFAULT
        )

        for p in model.parameters():
            p.requires_grad = False

        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(
            in_features,
            NUM_CLASSES,
        )

        head_parameters = model.classifier.parameters()

        def unfreeze_stage2():
            for p in model.features[-3:].parameters():
                p.requires_grad = True

        stage2_lr = 1e-4

    else:
        raise ValueError(model_name)

    return (
        model.to(DEVICE),
        head_parameters,
        unfreeze_stage2,
        stage2_lr,
    )

## 8. Training utilities

In [9]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

def topk_correct(logits, targets, k=5):
    k = min(k, logits.shape[1])
    return logits.topk(k, dim=1).indices.eq(
        targets.view(-1, 1)
    ).any(dim=1).sum().item()

def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    top1_correct = 0
    top5_correct = 0
    samples = 0

    for images, targets in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = model(images)
            loss = criterion(logits, targets)

        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        bs = images.size(0)

        total_loss += loss.item() * bs
        top1_correct += (
            logits.argmax(1) == targets
        ).sum().item()
        top5_correct += topk_correct(
            logits,
            targets,
            5,
        )
        samples += bs

    return {
        "loss": total_loss / samples,
        "top1": top1_correct / samples,
        "top5": top5_correct / samples,
    }

def train_stage(model, optimizer, epochs, stage_name):
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    history = []
    best_val = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        start = time.perf_counter()

        train_metrics = run_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
        )

        with torch.inference_mode():
            val_metrics = run_epoch(
                model,
                val_loader,
            )

        elapsed = time.perf_counter() - start

        row = {
            "stage": stage_name,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_top1": train_metrics["top1"],
            "val_loss": val_metrics["loss"],
            "val_top1": val_metrics["top1"],
            "val_top5": val_metrics["top5"],
            "seconds": elapsed,
        }

        history.append(row)

        print(
            f"[{stage_name}] {epoch}/{epochs} | "
            f"train={row['train_top1']:.3f} "
            f"val={row['val_top1']:.3f} "
            f"top5={row['val_top5']:.3f} | "
            f"{elapsed:.1f}s"
        )

        if row["val_top1"] > best_val:
            best_val = row["val_top1"]
            best_state = copy.deepcopy(
                model.state_dict()
            )

    model.load_state_dict(best_state)

    return history, best_val, best_state

## 9. Test evaluation

In [10]:
def evaluate(model):
    model.eval()

    y_true = []
    y_pred = []
    probabilities_all = []

    with torch.inference_mode():
        for images, targets in tqdm(
            test_loader,
            leave=False,
        ):
            images = images.to(
                DEVICE,
                non_blocking=True,
            )

            probabilities = torch.softmax(
                model(images),
                dim=1,
            )

            y_true.extend(targets.numpy())
            y_pred.extend(
                probabilities.argmax(1).cpu().numpy()
            )
            probabilities_all.append(
                probabilities.cpu()
            )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    probabilities = torch.cat(
        probabilities_all,
        dim=0,
    )

    top1 = accuracy_score(
        y_true,
        y_pred,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
    )

    top5_indices = probabilities.topk(
        min(5, NUM_CLASSES),
        dim=1,
    ).indices.numpy()

    top5 = np.mean([
        true in candidates
        for true, candidates in zip(
            y_true,
            top5_indices,
        )
    ])

    report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=list(range(NUM_CLASSES)),
            target_names=class_names,
            output_dict=True,
            zero_division=0,
        )
    ).T

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
    )

    return (
        top1,
        top5,
        macro_f1,
        report_df,
        cm,
    )

## 10. Running CNN architecture benchmark

In [11]:
STAGE1_EPOCHS = 3
STAGE2_EPOCHS = 5

results = []

for model_name in [
    "efficientnet_b0",
    "convnext_tiny",
    "mobilenet_v3_large",
]:
    print("\n" + "=" * 80)
    print("TRAINING:", model_name)
    print("=" * 80)

    (
        model,
        head_parameters,
        unfreeze_stage2,
        stage2_lr,
    ) = build_model(model_name)

    optimizer = torch.optim.AdamW(
        head_parameters,
        lr=1e-3,
        weight_decay=1e-4,
    )

    history1, val1, state1 = train_stage(
        model,
        optimizer,
        STAGE1_EPOCHS,
        "head",
    )

    unfreeze_stage2()

    optimizer = torch.optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters(),
        ),
        lr=stage2_lr,
        weight_decay=1e-4,
    )

    history2, val2, state2 = train_stage(
        model,
        optimizer,
        STAGE2_EPOCHS,
        "fine_tune",
    )

    if val1 > val2:
        model.load_state_dict(state1)
        selected_stage = "head"
        best_val = val1
    else:
        model.load_state_dict(state2)
        selected_stage = "fine_tune"
        best_val = val2

    (
        top1,
        top5,
        macro_f1,
        report_df,
        cm,
    ) = evaluate(model)

    model_report_dir = REPORT_DIR / model_name
    model_report_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.DataFrame(
        history1 + history2
    ).to_csv(
        model_report_dir / "training_history.csv",
        index=False,
    )

    report_df.to_csv(
        model_report_dir / "classification_report.csv"
    )

    fig, ax = plt.subplots(figsize=(16, 16))

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    ).plot(
        ax=ax,
        xticks_rotation=90,
        colorbar=False,
        values_format="d",
    )

    ax.set_title(
        f"{model_name} — Test Confusion Matrix"
    )

    plt.tight_layout()

    plt.savefig(
        model_report_dir / "confusion_matrix.png",
        dpi=160,
    )

    plt.close()

    model_path = (
        MODEL_DIR
        / f"{model_name}_compcars_baseline.pt"
    )

    torch.save(
        {
            "architecture": model_name,
            "state_dict": model.state_dict(),
            "classes": class_names,
            "num_classes": NUM_CLASSES,
            "selected_stage": selected_stage,
            "validation_top1": float(best_val),
            "test_top1": float(top1),
            "test_top5": float(top5),
            "test_macro_f1": float(macro_f1),
            "gpu": GPU_NAME,
        },
        model_path,
    )

    metrics = {
        "Model": model_name,
        "Validation Top-1": float(best_val),
        "Test Top-1": float(top1),
        "Test Top-5": float(top5),
        "Macro F1": float(macro_f1),
        "GPU": GPU_NAME,
    }

    results.append(metrics)

    with (
        model_report_dir / "metrics.json"
    ).open("w", encoding="utf-8") as f:
        json.dump(
            metrics,
            f,
            indent=2,
        )

    print(metrics)

    del model
    torch.cuda.empty_cache()


TRAINING: efficientnet_b0
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\elesh/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 62.3MB/s]


[head] 1/3 | train=0.188 val=0.315 top5=0.690 | 68.0s


[head] 2/3 | train=0.426 val=0.410 top5=0.775 | 56.1s


[head] 3/3 | train=0.556 val=0.445 top5=0.815 | 64.8s


...

[fine_tune] 2/5 | train=0.811 val=0.630 top5=0.915 | 52.9s


[fine_tune] 3/5 | train=0.879 val=0.670 top5=0.930 | 53.0s


[fine_tune] 4/5 | train=0.919 val=0.655 top5=0.930 | 52.3s


[fine_tune] 5/5 | train=0.953 val=0.695 top5=0.945 | 53.4s


{'Model': 'mobilenet_v3_large', 'Validation Top-1': 0.695, 'Test Top-1': 0.735, 'Test Top-5': 0.975, 'Macro F1': 0.731191591170828, 'GPU': 'NVIDIA GeForce RTX 5060 Laptop GPU'}


## 11. Comparing the new CNNs

In [12]:
comparison_df = (
    pd.DataFrame(results)
    .sort_values(
        "Test Top-1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(comparison_df)

comparison_df.to_csv(
    REPORT_DIR / "cnn_architecture_comparison.csv",
    index=False,
)

,Model,Validation Top-1,Test Top-1,Test Top-5,Macro F1,GPU
0,convnext_tiny,0.840,0.835,1.000,0.833641,NVIDIA GeForce RTX 5060 Laptop GPU
1,mobilenet_v3_large,0.695,0.735,0.975,0.731192,NVIDIA GeForce RTX 5060 Laptop GPU
2,efficientnet_b0,0.555,0.500,0.875,0.492892,NVIDIA GeForce RTX 5060 Laptop GPU
